# Notebook 2: Tokenization & Embeddings (JAX)

## Learning Objectives
- Understand tokenization
- Implement token embeddings in JAX/Flax
- Implement Rotary Position Embedding (RoPE)
- Understand JAX functional paradigm

## Task 1: JAX Basics

### HINT: JAX vs PyTorch
```python
# PyTorch
model = MyModel()
output = model(input)

# JAX (Flax)
model = MyModel()
params = model.init(rng, input)
output = model.apply(params, input)
```

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
from flax import linen as nn
from flax.linen import initializers

# JAX is functional - no in-place updates
# Need to manage state explicitly

print("JAX devices:", jax.devices())
print("JAX version:", jax.__version__)

## Task 2: Token Embeddings in Flax

In [ ]:
class TokenEmbedding(nn.Module):
    """Token embedding in Flax"""
    vocab_size: int
    hidden_size: int
    
    @nn.compact
    def __call__(self, input_ids):
        # Use nn.embed for embeddings
        embedding = self.param(
            'embedding',
            initializers.normal(stddev=0.02),
            (self.vocab_size, self.hidden_size)
        )
        return embedding[input_ids]

# Test
vocab_size = 50000
hidden_size = 512
seq_len = 8
batch_size = 2

model = TokenEmbedding(vocab_size=vocab_size, hidden_size=hidden_size)
rng = random.PRNGKey(42)
input_ids = jax.random.randint(rng, (batch_size, seq_len), 0, vocab_size)

params = model.init(rng, input_ids)
output = model.apply(params, input_ids)

print(f"Input shape: {input_ids.shape}")
print(f"Output shape: {output.shape}")

## Task 3: RoPE in JAX

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """RoPE in Flax"""
    dim: int
    max_seq_len: int = 512
    base: float = 10000.0
    
    def setup(self):
        # Precompute inverse frequencies
        inv_freq = 1.0 / (self.base ** (jnp.arange(0, self.dim, 2) / self.dim))
        self.inv_freq = self.variable('rope', 'inv_freq', lambda _: inv_freq)
    
    def __call__(self, seq_len):
        positions = jnp.arange(seq_len)
        angles = positions[:, None] * self.inv_freq[None, :]
        cos = jnp.cos(angles)
        sin = jnp.sin(angles)
        return cos, sin

rope = RotaryPositionalEmbedding(dim=64, max_seq_len=512)
params = rope.init(rng, 10)
cos, sin = rope.apply(params, 10)
print(f"Cos shape: {cos.shape}")
print(f"Sin shape: {sin.shape}")

## Task 4: Apply RoPE to Q/K

In [ ]:
def apply_rope(x, cos, sin):
    """Apply rotary position embedding"""
    head_dim = x.shape[-1]
    x1 = x[..., :head_dim // 2]
    x2 = x[..., head_dim // 2:]
    
    # Reshape for broadcasting
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    
    return jnp.concatenate([
        x1 * cos + x2 * sin,
        x2 * cos - x1 * sin
    ], axis=-1)

# Test RoPE
batch, heads, seq, head_dim = 2, 8, 16, 64
x = random.normal(rng, (batch, heads, seq, head_dim))
x_rope = apply_rope(x, cos, sin)
print(f"Input shape: {x.shape}")
print(f"After RoPE: {x_rope.shape}")

## Verification

1. ✅ Can create Flax modules
2. ✅ Can implement token embeddings
3. ✅ Can implement RoPE
4. ✅ Understand JAX functional paradigm